# Data Challenge HUN 2026: predicción de demanda en urgencias

## 1. Introducción y contexto

Este notebook documenta una metodología para estudiar el flujo horario de pacientes en el servicio de urgencias del Hospital Universitario Nacional (HUN). Pronosticar la actividad esperada puede apoyar la planificación operacional, por ejemplo, al anticipar horas que podrían requerir mayor coordinación de personal, espacios o recursos. El propósito de este análisis es aportar evidencia cuantitativa para la toma de decisiones; no sustituye la validación clínica ni las decisiones institucionales.

La variable analizada representa episodios inferidos a partir de registros OMOP. Por tanto, un episodio inferido no equivale necesariamente a una llegada física confirmada al servicio de urgencias. Esta distinción es central para interpretar los resultados y para evitar que una predicción estadísticamente precisa se presente como una medición directa de ingresos hospitalarios.

El flujo comienza con la preparación del entorno computacional y continúa con la conexión a la fuente de datos. En las siguientes celdas se documentan progresivamente las decisiones de datos, modelado y evaluación.


## 2. Objetivo del modelo

El objetivo es pronosticar cuántos episodios inferidos comienzan durante una hora determinada. La unidad de predicción es, por tanto, una hora y la variable objetivo `demanda` representa un conteo numérico de episodios inferidos, no necesariamente llegadas físicas confirmadas.

Se plantea como un problema de regresión supervisada porque el modelo aprende a asociar variables temporales y observaciones históricas con un valor cuantitativo. La regresión permite estimar una cantidad, como 35,7 episodios esperados, y evaluar la distancia entre la predicción y el valor observado mediante MAE y RMSE. La clasificación sería apropiada si el objetivo principal fuera asignar cada hora a categorías discretas, por ejemplo, actividad alta o no alta; aquí esa transformación se utiliza posteriormente solo para explorar alertas.

Esta formulación permite conservar información sobre la magnitud de la demanda y, después, convertir la predicción continua en una señal binaria mediante umbrales explícitos. La siguiente etapa presenta las herramientas y la arquitectura empleadas para implementar el flujo completo.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import psycopg
import sqlalchemy

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PostgreSQL driver:", psycopg.__version__)
print("Entorno listo para el Data Challenge HUN 2026")

## 3. Herramientas y arquitectura

La implementación utiliza Python como lenguaje de análisis, pandas y NumPy para manipulación y cálculo numérico, scikit-learn para los modelos y métricas, matplotlib para la exploración visual y SQLAlchemy/psycopg para acceder a PostgreSQL. `joblib` se emplea posteriormente para persistir modelos y `Path` para organizar archivos de salida.

La arquitectura analítica sigue el flujo: **PostgreSQL → pandas → análisis exploratorio → ingeniería de variables → entrenamiento → validación → evaluación → resultados para Power BI**. En este notebook, Power BI representa una posible capa de consumo de resultados; la exportación observada es un archivo CSV y no una conexión directa con Power BI.

Esta primera celda verifica versiones y disponibilidad del entorno. No produce un modelo ni modifica los datos; establece las condiciones de reproducibilidad de las etapas siguientes.


In [ ]:
import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

password = getpass.getpass("Contraseña de PostgreSQL: ")

url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="hun_analitica"
)

engine = create_engine(url)

with engine.connect() as conexion:
    resultado = conexion.execute(text("SELECT version()"))
    print("✅ Conexión exitosa con PostgreSQL")
    print(resultado.scalar().split(",")[0])

del password

## 4. Conexión y extracción de datos

La conexión se realiza contra una instancia local de PostgreSQL mediante SQLAlchemy y el controlador `psycopg`. La contraseña se solicita de forma interactiva y se elimina de la variable al finalizar la conexión; no debe registrarse en el notebook ni en archivos compartidos. La consulta lee la tabla `urgencias.demanda_horaria`, ordenada por `fecha_hora`, y carga su resultado en un `DataFrame` de pandas.

Las columnas extraídas son: `fecha_hora` (marca temporal), `demanda` (episodios inferidos de la hora), `hora`, `dia_semana`, `mes`, `anio` y `es_fin_semana`. La tabla se utiliza como fuente de la variable objetivo y de las variables temporales/calendario disponibles. No se asumen columnas adicionales.

La conversión explícita de `fecha_hora` a tipo datetime permite filtrar cronológicamente y construir variables rezagadas. A continuación se verifica la estructura y calidad inicial del conjunto leído.


In [ ]:
query = """
SELECT fecha_hora, demanda, hora, dia_semana,
       mes, anio, es_fin_semana
FROM urgencias.demanda_horaria
ORDER BY fecha_hora
"""

df = pd.read_sql(query, engine)

df["fecha_hora"] = pd.to_datetime(df["fecha_hora"])

print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nEstadísticas de demanda:")
print(df["demanda"].describe())

## 5. Calidad y análisis exploratorio

Este bloque revisa dimensiones, tipos de datos, valores nulos y estadísticas descriptivas de `demanda`. Es una comprobación básica de calidad: permite detectar problemas de tipado, ausencia de observaciones o valores incompatibles antes de entrenar modelos. La salida observada contiene 31.051 filas y siete columnas, sin valores nulos en las columnas inspeccionadas.

La media observada de demanda es 25,81 episodios por hora y la mediana es 19. La media superior a la mediana indica una distribución asimétrica hacia valores altos, consistente con períodos de actividad elevada que elevan el promedio. El máximo observado es 165, por lo que las horas excepcionales pueden influir de forma importante en métricas sensibles a errores grandes.

El histograma siguiente permite visualizar esa distribución y prepara la decisión metodológica de separar el tiempo en entrenamiento, validación y prueba.


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df["demanda"], bins=50, edgecolor="black", alpha=0.7)

plt.axvline(
    df["demanda"].mean(),
    color="red",
    linestyle="--",
    label=f"Media: {df['demanda'].mean():.1f}"
)

plt.axvline(
    df["demanda"].median(),
    color="green",
    linestyle="--",
    label=f"Mediana: {df['demanda'].median():.1f}"
)

plt.title("Distribución de actividad horaria - Urgencias HUN")
plt.xlabel("Episodios inferidos por hora")
plt.ylabel("Frecuencia de horas")
plt.legend()
plt.tight_layout()
plt.show()

## 6. División cronológica

La separación respeta el orden temporal: entrenamiento con 2023–2024, validación con 2025 y prueba con las observaciones desde enero de 2026, que en la salida disponible llegan hasta el 17 de julio de 2026 a las 18:00. La salida registrada muestra 17.544 filas de entrenamiento, 8.760 de validación y 4.747 de prueba.

Esta estrategia simula el uso real: se entrena con el pasado, se seleccionan decisiones con un período posterior y se reserva un período aún más reciente para evaluar. El *data leakage* ocurre cuando información del futuro respecto del instante de predicción influye en el entrenamiento o en las variables predictoras. Evitarlo es necesario para que las métricas representen razonablemente el desempeño esperado fuera de muestra.

Con los períodos definidos, primero se establece una referencia sencilla y transparente contra la cual comparar los modelos de Machine Learning.


In [ ]:
train = df[df["fecha_hora"] < "2025-01-01"].copy()

val = df[
    (df["fecha_hora"] >= "2025-01-01") &
    (df["fecha_hora"] < "2026-01-01")
].copy()

test = df[df["fecha_hora"] >= "2026-01-01"].copy()

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", val),
    ("Prueba", test)
]:
    print(
        nombre,
        "| Filas:", len(datos),
        "| Media:", round(datos["demanda"].mean(), 2),
        "| Inicio:", datos["fecha_hora"].min(),
        "| Fin:", datos["fecha_hora"].max()
    )

Nuestro Modelo 1 será de regresión, no de clasificación. Y la razón está directamente relacionada con el problema que queremos resolver en el hospital.
1. ¿Por qué regresión?
Porque queremos predecir cuántos episodios de urgencias se registrarán en una hora determinada.
Nuestra variable objetivo es demanda, que es numérica.
Por ejemplo:
Hora	Predicción
Lunes, 8:00 a. m.	42 episodios
Lunes, 9:00 a. m.	57 episodios
Lunes, 10:00 a. m.	49 episodios
Son ejemplos ilustrativos, no predicciones reales.
La regresión permite estimar una cantidad numérica. Aunque los episodios se cuentan con números enteros, podemos entrenar una regresión y evaluar sus resultados mediante MAE y RMSE. Más adelante también podríamos explorar modelos específicos para conteos.

## 7. Modelo baseline histórico

El baseline calcula, para cada combinación de `dia_semana` y `hora`, el promedio de `demanda` observado en el entrenamiento (2023–2024). Ese patrón se cruza con 2025 para producir una predicción por hora. Es una referencia interpretable: supone que el comportamiento típico del mismo día de semana y hora se mantiene en el período siguiente.

El error absoluto medio (MAE) se define como

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|,$$

y el error cuadrático medio en raíz (RMSE) como

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}.$$

Ambas métricas se interpretan en episodios por hora. El MAE resume el error típico; el RMSE penaliza más los errores grandes, por lo que ayuda a evidenciar dificultades en horas atípicas. En la salida observada, el baseline obtiene MAE 8,23 y RMSE 12,36, sin predicciones faltantes en validación. La celda siguiente repite este cálculo, sin cambiar la referencia utilizada.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

patrones = (
    train.groupby(["dia_semana", "hora"])["demanda"]
    .mean()
    .rename("prediccion")
    .reset_index()
)

val_base = val.merge(
    patrones,
    on=["dia_semana", "hora"],
    how="left"
)

mae = mean_absolute_error(
    val_base["demanda"],
    val_base["prediccion"]
)

rmse = np.sqrt(
    mean_squared_error(
        val_base["demanda"],
        val_base["prediccion"]
    )
)

print("MAE Baseline:", round(mae, 2))
print("RMSE Baseline:", round(rmse, 2))
print("Predicciones faltantes:", val_base["prediccion"].isna().sum())

### Repetición del cálculo del baseline

Esta celda reproduce el baseline histórico y sus métricas con el mismo procedimiento. Se conserva para respetar el orden y el contenido original del análisis; no constituye un segundo método independiente. Su resultado coincide con MAE 8,23 y RMSE 12,36, y confirma la referencia antes de introducir un modelo de árboles.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

patrones = (
    train.groupby(["dia_semana", "hora"])["demanda"]
    .mean()
    .rename("prediccion")
    .reset_index()
)

val_base = val.merge(
    patrones,
    on=["dia_semana", "hora"],
    how="left"
)

mae = mean_absolute_error(
    val_base["demanda"],
    val_base["prediccion"]
)

rmse = np.sqrt(
    mean_squared_error(
        val_base["demanda"],
        val_base["prediccion"]
    )
)

print("MAE Baseline:", round(mae, 2))
print("RMSE Baseline:", round(rmse, 2))
print("Predicciones faltantes:", val_base["prediccion"].isna().sum())

No incluimos anio por ahora: queremos evaluar si los patrones de calendario se generalizan a 2025, sin introducir una variable que el modelo no haya visto con ese valor durante el entrenamiento.

## 8. Random Forest inicial

`RandomForestRegressor` combina muchos árboles de decisión entrenados sobre variaciones de los datos y de las variables. La predicción final es el promedio de sus predicciones, lo que suele reducir la variabilidad de un árbol individual. Aquí se utilizan `hora`, `dia_semana`, `mes` y `es_fin_semana`; no se incluye `anio`, porque se busca evaluar la generalización de patrones de calendario a 2025 sin introducir un valor anual no observado durante el entrenamiento.

Los hiperparámetros son `n_estimators=200`, `min_samples_leaf=5`, `random_state=42` y `n_jobs=-1`. Los dos primeros controlan el número de árboles y el tamaño mínimo de las hojas; la semilla favorece la reproducibilidad y `n_jobs` permite paralelizar el ajuste.

La salida almacenada para la ejecución equivalente reporta MAE 8,51 y RMSE 12,75, frente a 8,23 y 12,36 del baseline. En esa validación, el Random Forest inicial no supera la referencia histórica. Se conserva la comparación como evidencia observada, no como una conclusión universal sobre el algoritmo.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

variables = [
    "hora",
    "dia_semana",
    "mes",
    "es_fin_semana"
]

X_train = train[variables]
y_train = train["demanda"]

X_val = val[variables]
y_val = val["demanda"]

rf = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_val)

mae_rf = mean_absolute_error(y_val, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, pred_rf))

print("MAE Random Forest:", round(mae_rf, 2))
print("RMSE Random Forest:", round(rmse_rf, 2))
print("MAE Baseline:", round(mae, 2))
print("RMSE Baseline:", round(rmse, 2))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

variables = [
    "hora",
    "dia_semana",
    "mes",
    "es_fin_semana"
]

X_train = train[variables]
y_train = train["demanda"]

X_val = val[variables]
y_val = val["demanda"]

rf = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_val)

mae_rf = mean_absolute_error(y_val, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, pred_rf))

print("MAE Random Forest:", round(mae_rf, 2))
print("RMSE Random Forest:", round(rmse_rf, 2))
print("MAE Baseline:", round(mae, 2))
print("RMSE Baseline:", round(rmse, 2))

## 9. Análisis del error y baseline adaptativo

Este bloque resume por mes el promedio real, el promedio predicho por el baseline histórico y el error con signo (`prediccion - demanda`). El objetivo es identificar cambios estacionales o períodos en los que el patrón histórico sobreestima o subestima sistemáticamente. La salida observada muestra errores positivos en varios meses, aunque la magnitud y el significado operacional deben interpretarse junto con el número de horas y la variabilidad de cada grupo.

El análisis motiva un baseline adaptativo basado en las cuatro semanas anteriores. Este enfoque intenta incorporar el nivel reciente de actividad manteniendo una regla simple y disponible retrospectivamente. La comparación posterior permite distinguir si la adaptación mejora realmente el desempeño o si añade variabilidad sin beneficio.


In [ ]:
analisis = val_base.copy()

analisis["error"] = (
    analisis["prediccion"] - analisis["demanda"]
)

resumen_mensual = (
    analisis.groupby("mes")
    .agg(
        real=("demanda", "mean"),
        predicho=("prediccion", "mean"),
        error_medio=("error", "mean")
    )
    .round(2)
)

print(resumen_mensual)

### Construcción inicial del baseline adaptativo

Esta celda ordena temporalmente los registros y calcula una media móvil de cuatro observaciones semanales desplazadas 168 horas. Se documenta como paso intermedio de construcción; la evaluación completa del baseline adaptativo se realiza en la celda siguiente. El desplazamiento evita usar la demanda de la hora que se intenta predecir, siempre que los registros estén completos y correctamente ordenados.


In [ ]:
df_adapt = df.sort_values("fecha_hora").copy()

df_adapt["pred_adaptativa"] = (
    df_adapt["demanda"]
    .shift(168)
    .rolling(window=4, min_periods=4)
    .mean()
)

La comparación observada en 2025 reporta MAE 8,23 y RMSE 12,36 para el baseline histórico, frente a MAE 8,38 y RMSE 13,00 para el adaptativo. En este experimento, el promedio histórico fue mejor según ambas métricas. Esto sugiere que la información reciente de cuatro semanas, tal como fue implementada, no garantizó una mejora; no permite concluir que todas las estrategias adaptativas sean ineficaces. El siguiente paso es incorporar rezagos directamente al modelo supervisado.


In [ ]:
df_adapt = df.sort_values("fecha_hora").copy()

df_adapt["pred_adaptativa"] = (
    df_adapt["demanda"].shift(168) +
    df_adapt["demanda"].shift(336) +
    df_adapt["demanda"].shift(504) +
    df_adapt["demanda"].shift(672)
) / 4

val_adapt = df_adapt[
    (df_adapt["fecha_hora"] >= "2025-01-01") &
    (df_adapt["fecha_hora"] < "2026-01-01")
].copy()

mae_adapt = mean_absolute_error(
    val_adapt["demanda"],
    val_adapt["pred_adaptativa"]
)

rmse_adapt = np.sqrt(
    mean_squared_error(
        val_adapt["demanda"],
        val_adapt["pred_adaptativa"]
    )
)

print("MAE Baseline histórico:", round(mae, 2))
print("RMSE Baseline histórico:", round(rmse, 2))
print("MAE Baseline adaptativo:", round(mae_adapt, 2))
print("RMSE Baseline adaptativo:", round(rmse_adapt, 2))

### Incorporación de variables temporales al modelo predictivo

Después de evaluar tres enfoques de predicción, encontramos que el **baseline histórico obtuvo el menor error (MAE = 8,23)**, superando a Random Forest y al baseline adaptativo. Esto sugiere que los patrones de calendario son útiles, pero que incorporar información reciente de forma sencilla no garantiza mejores resultados. Por ello, construiremos nuevas variables temporales, como la actividad del día anterior (`lag_24`), de la semana anterior (`lag_168`) y el promedio de las últimas 24 horas (`media_24h`). Con estas variables entrenaremos un **Random Forest mejorado**, buscando capturar tanto los patrones habituales como los cambios recientes en la actividad hospitalaria, sin utilizar información futura. Posteriormente, compararemos su desempeño con el baseline para determinar si aporta una mejora real.

## 10. Ingeniería de variables temporales

Se construyen tres variables usando únicamente observaciones anteriores: `lag_24` es la demanda observada 24 horas antes y aproxima la misma hora del día previo; `lag_168` es la demanda observada 168 horas antes y aproxima la misma hora de la semana previa; `media_24h` es la media de las 24 horas anteriores, excluyendo la hora objetivo mediante `shift(1)`. Después se eliminan las filas iniciales que no tienen historial suficiente.

Cada variable aporta una señal distinta: el rezago de 24 horas puede capturar persistencia diaria, el de 168 horas puede capturar repetición semanal y la media reciente puede representar el nivel general de actividad. En una operación real, estas variables solo estarían disponibles si los registros de las horas anteriores se reciben oportunamente, con continuidad temporal y sin revisiones que alteren retrospectivamente los valores.

La salida observada queda con 30.883 filas, 10 columnas y cero valores nulos después del `dropna`. Con las variables creadas, se entrena el Random Forest mejorado manteniendo la división cronológica.


In [ ]:
df_ml = df.sort_values("fecha_hora").copy()

df_ml["lag_24"] = df_ml["demanda"].shift(24)
df_ml["lag_168"] = df_ml["demanda"].shift(168)

df_ml["media_24h"] = (
    df_ml["demanda"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df_ml = df_ml.dropna().copy()

print("Dimensiones:", df_ml.shape)
print("Valores nulos:", df_ml.isnull().sum().sum())

display(
    df_ml[
        ["fecha_hora", "demanda", "lag_24", "lag_168", "media_24h"]
    ].head()
)

## 11. Random Forest mejorado

El modelo mejorado conserva las variables de calendario y añade `lag_24`, `lag_168` y `media_24h`. Se entrena con 2023–2024 y se valida con 2025 usando los mismos hiperparámetros del Random Forest inicial (`n_estimators=200`, `min_samples_leaf=5`, `random_state=42`, `n_jobs=-1`).

La salida observada reporta MAE 7,65 y RMSE 11,66, frente a MAE 8,23 del baseline y MAE 8,51 del Random Forest original. En esta validación, la incorporación de historial reciente mejora el error observado. Esta conclusión se limita al período y al procedimiento evaluados; todavía debe examinarse dónde se concentran los errores y si existe un patrón operativo relevante.


In [ ]:
variables_ml = [
    "hora",
    "dia_semana",
    "mes",
    "es_fin_semana",
    "lag_24",
    "lag_168",
    "media_24h"
]

train_ml = df_ml[df_ml["fecha_hora"] < "2025-01-01"]
val_ml = df_ml[
    (df_ml["fecha_hora"] >= "2025-01-01") &
    (df_ml["fecha_hora"] < "2026-01-01")
]

rf_mejorado = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_mejorado.fit(
    train_ml[variables_ml],
    train_ml["demanda"]
)

pred_mejorado = rf_mejorado.predict(val_ml[variables_ml])

mae_mejorado = mean_absolute_error(
    val_ml["demanda"], pred_mejorado
)

rmse_mejorado = np.sqrt(
    mean_squared_error(val_ml["demanda"], pred_mejorado)
)

print("MAE Baseline:", round(mae, 2))
print("MAE RF original:", round(mae_rf, 2))
print("MAE RF mejorado:", round(mae_mejorado, 2))
print("RMSE RF mejorado:", round(rmse_mejorado, 2))

## 12. Interpretabilidad mediante permutation importance

La importancia por permutación mide cuánto empeora el desempeño cuando se desordena una variable en el conjunto de validación, manteniendo las demás. Con `scoring="neg_mean_absolute_error"`, un valor positivo indica que permutar la variable aumenta el error y, por tanto, que la variable aporta capacidad predictiva en este experimento. Un valor cercano a cero indica poca contribución incremental bajo esta evaluación; incluso puede haber valores negativos por variación muestral.

La salida observada ordena `hora` como la variable con mayor importancia media, seguida por `lag_168` y `media_24h`; `mes` presenta una importancia cercana a cero. Estos resultados describen asociación predictiva, no causalidad: una variable importante no demuestra que cause la demanda ni que modificarla produzca un efecto hospitalario. Además, variables correlacionadas pueden repartirse la importancia.


In [ ]:
from sklearn.inspection import permutation_importance

importancias = permutation_importance(
    rf_mejorado,
    val_ml[variables_ml],
    val_ml["demanda"],
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

resultado_importancias = (
    pd.DataFrame({
        "variable": variables_ml,
        "importancia": importancias.importances_mean
    })
    .sort_values("importancia", ascending=False)
)

display(resultado_importancias)

## 15. Evaluación con datos de 2026

Esta celda ajusta un **Random Forest** con variables temporales usando todos los datos anteriores a 2026 y evalúa sobre el período de prueba disponible de 2026. La salida observada contiene 4.747 horas, MAE 6,71, RMSE 9,74, demanda real media 19,86 y predicha media 21,48. La celda siguiente compara este resultado con un baseline construido también sin datos objetivo de 2026.

Es importante distinguir este Random Forest evaluado en 2026 del `HistGradientBoosting` validado en 2025: sus métricas no son intercambiables ni deben atribuirse al segundo modelo. Además, el conjunto de prueba 2026 ya fue consultado durante este análisis; por ello, la evaluación debe tratarse como retrospectiva y no como una prueba futura completamente ciega.


In [ ]:
train_final = df_ml[
    df_ml["fecha_hora"] < "2026-01-01"
].copy()

test_final = df_ml[
    df_ml["fecha_hora"] >= "2026-01-01"
].copy()

rf_final = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(
    train_final[variables_ml],
    train_final["demanda"]
)

pred_final = rf_final.predict(
    test_final[variables_ml]
)

mae_final = mean_absolute_error(
    test_final["demanda"],
    pred_final
)

rmse_final = np.sqrt(
    mean_squared_error(
        test_final["demanda"],
        pred_final
    )
)

print("Horas evaluadas:", len(test_final))
print("MAE Test 2026:", round(mae_final, 2))
print("RMSE Test 2026:", round(rmse_final, 2))
print("Demanda real promedio:", round(test_final["demanda"].mean(), 2))
print("Demanda predicha promedio:", round(pred_final.mean(), 2))

La comparación observada para 2026 reporta MAE 10,64 y RMSE 15,64 para el baseline, frente a MAE 6,71 y RMSE 9,74 para el Random Forest evaluado en 2026, con una mejora de MAE calculada de 36,89 %. Estas cifras describen la comparación implementada y no validan por sí solas un despliegue en producción. La siguiente etapa examina cómo cambia el error en rangos de actividad.


In [ ]:
patrones_finales = (
    train_final.groupby(["dia_semana", "hora"])["demanda"]
    .mean()
    .rename("pred_baseline")
    .reset_index()
)

test_comparacion = test_final.merge(
    patrones_finales,
    on=["dia_semana", "hora"],
    how="left"
)

mae_base_2026 = mean_absolute_error(
    test_comparacion["demanda"],
    test_comparacion["pred_baseline"]
)

rmse_base_2026 = np.sqrt(
    mean_squared_error(
        test_comparacion["demanda"],
        test_comparacion["pred_baseline"]
    )
)

mejora_mae = (
    (mae_base_2026 - mae_final) / mae_base_2026
) * 100

print("MAE Baseline 2026:", round(mae_base_2026, 2))
print("RMSE Baseline 2026:", round(rmse_base_2026, 2))
print("MAE Random Forest 2026:", round(mae_final, 2))
print("RMSE Random Forest 2026:", round(rmse_final, 2))
print("Mejora MAE (%):", round(mejora_mae, 2))
print("Predicciones faltantes:", test_comparacion["pred_baseline"].isna().sum())

## 14. Análisis de períodos de actividad elevada

Este bloque agrupa la evaluación 2026 por demanda real: 0–20, 21–40, 41–60 y más de 60 episodios. La salida observada muestra MAE 4,89, 8,56, 9,40 y 21,45, respectivamente; en el rango superior la demanda real media es 76,39 y la predicha 55,89. Por tanto, el error crece de forma marcada durante las horas excepcionalmente altas y el modelo tiende a subestimarlas.

La limitación no debe generalizarse a toda actividad elevada: en niveles habituales el modelo presenta errores menores, mientras que los picos extremos son menos frecuentes y más difíciles de anticipar con las variables disponibles. Este análisis separa desempeño promedio de comportamiento en escenarios críticos, lo que es especialmente relevante para alertas operacionales.


In [ ]:
evaluacion = test_final.copy()

evaluacion["prediccion"] = pred_final

evaluacion["error_abs"] = abs(
    evaluacion["demanda"] - evaluacion["prediccion"]
)

evaluacion["rango"] = pd.cut(
    evaluacion["demanda"],
    bins=[-1, 20, 40, 60, float("inf")],
    labels=["0-20", "21-40", "41-60", "Más de 60"]
)

resumen_errores = (
    evaluacion.groupby("rango", observed=True)
    .agg(
        horas=("demanda", "count"),
        demanda_real=("demanda", "mean"),
        demanda_predicha=("prediccion", "mean"),
        mae=("error_abs", "mean")
    )
    .round(2)
)

display(resumen_errores)

## 13. HistGradientBoosting

`HistGradientBoostingRegressor` construye árboles pequeños de manera secuencial: cada nuevo árbol intenta corregir los errores acumulados por los anteriores. La discretización por histogramas puede hacer eficiente el entrenamiento en conjuntos tabulares. En este notebook se usa `max_iter=200`, `max_leaf_nodes=15`, `learning_rate=0.05`, `l2_regularization=1.0` y `random_state=42`.

En la validación de 2025, la salida observada reporta MAE 7,52 y RMSE 11,43, frente a MAE 7,65 y RMSE 11,66 del Random Forest mejorado. HGB presenta menor error en ambas métricas dentro de esta comparación. Esta conclusión corresponde exclusivamente a validación 2025; no debe confundirse con el Random Forest que posteriormente se evaluó sobre 2026.

Las celdas siguientes profundizan en el comportamiento por rango de demanda y por hora antes de guardar el HGB y construir alertas exploratorias.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    max_iter=200,
    max_leaf_nodes=15,
    learning_rate=0.05,
    l2_regularization=1.0,
    random_state=42
)

hgb.fit(
    train_ml[variables_ml],
    train_ml["demanda"]
)

pred_hgb = hgb.predict(val_ml[variables_ml])

mae_hgb = mean_absolute_error(
    val_ml["demanda"], pred_hgb
)

rmse_hgb = np.sqrt(
    mean_squared_error(val_ml["demanda"], pred_hgb)
)

print("MAE RF mejorado:", round(mae_mejorado, 2))
print("RMSE RF mejorado:", round(rmse_mejorado, 2))
print("MAE HistGradientBoosting:", round(mae_hgb, 2))
print("RMSE HistGradientBoosting:", round(rmse_hgb, 2))

### Error por rango de demanda en la validación de 2025

La tabla compara Random Forest mejorado y HistGradientBoosting dentro de los rangos de demanda real. En la salida observada, HGB mejora ligeramente el MAE en 21–40, 41–60 y más de 60, mientras que ambos son iguales en 0–20 a dos decimales. En el rango superior, ambos modelos subestiman: la demanda real media es 80,39 y las predicciones medias son 58,78 y 58,67.

Este resultado refuerza que una métrica global puede ocultar fallas en episodios críticos. El análisis por hora permite localizar cuándo ocurren los sesgos y orientar futuras variables o validaciones.


In [ ]:
comparacion = val_ml.copy()

comparacion["pred_rf"] = pred_mejorado
comparacion["pred_hgb"] = pred_hgb

comparacion["error_rf"] = abs(
    comparacion["demanda"] - comparacion["pred_rf"]
)

comparacion["error_hgb"] = abs(
    comparacion["demanda"] - comparacion["pred_hgb"]
)

comparacion["rango"] = pd.cut(
    comparacion["demanda"],
    bins=[-1, 20, 40, 60, float("inf")],
    labels=["0-20", "21-40", "41-60", "Más de 60"]
)

resumen = (
    comparacion.groupby("rango", observed=True)
    .agg(
        horas=("demanda", "count"),
        mae_rf=("error_rf", "mean"),
        mae_hgb=("error_hgb", "mean"),
        pred_rf=("pred_rf", "mean"),
        pred_hgb=("pred_hgb", "mean"),
        demanda_real=("demanda", "mean")
    )
    .round(2)
)

display(resumen)

### Error por hora y sesgo horario

La tabla resume, para cada hora del día, la demanda real media, la predicción de HGB y el MAE. Los errores observados son mayores en las horas de actividad diurna, especialmente alrededor de las 8–11, que concentran niveles reales más altos. Esta diferencia puede reflejar tanto mayor variabilidad como dificultad para anticipar cambios rápidos.

La siguiente tabla calcula el sesgo medio `predicho - real`: valores positivos indican sobreestimación promedio y negativos subestimación promedio. Estas asociaciones son descriptivas y no prueban una causa; sirven para identificar segmentos que requieren seguimiento.


In [ ]:
errores_hora = (
    comparacion.groupby("hora")
    .agg(
        horas=("demanda", "count"),
        real=("demanda", "mean"),
        predicho=("pred_hgb", "mean"),
        mae=("error_hgb", "mean")
    )
    .round(2)
)

display(errores_hora)

La salida de sesgo horario muestra, entre otros valores, subestimación promedio a las horas 9–11 y sobreestimación a las 12 y 16–18. Debe interpretarse conjuntamente con los errores absolutos y con la distribución de casos, no como una regla automática de corrección. Con este diagnóstico se pasa a la persistencia del modelo y a la traducción exploratoria de predicciones en alertas.


In [ ]:
comparacion["sesgo_hgb"] = (
    comparacion["pred_hgb"] - comparacion["demanda"]
)

sesgo_hora = (
    comparacion.groupby("hora")
    .agg(
        real=("demanda", "mean"),
        predicho=("pred_hgb", "mean"),
        sesgo=("sesgo_hgb", "mean")
    )
    .round(2)
)

display(sesgo_hora)

## 17. Persistencia e integración

Esta celda guarda con `joblib` el modelo `hgb` junto con las variables utilizadas, los períodos de entrenamiento y validación, las métricas de validación y el horizonte de una hora. La persistencia permite reutilizar el artefacto sin volver a entrenarlo, siempre que se preserve un entorno compatible y se verifique la procedencia del archivo.

El objeto guardado corresponde al HistGradientBoosting validado en 2025, no al Random Forest evaluado en 2026. Esta distinción debe mantenerse en cualquier integración posterior con tableros o procesos operacionales. La exportación CSV de la última celda documenta resultados retrospectivos para consumo analítico.


In [ ]:
from pathlib import Path
import joblib

carpeta = Path("resultados/modelos")
carpeta.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "modelo": hgb,
        "variables": variables_ml,
        "periodo_entrenamiento": "2023-2024",
        "periodo_validacion": "2025",
        "mae_validacion": mae_hgb,
        "rmse_validacion": rmse_hgb,
        "horizonte": "1 hora",
        "objetivo": "episodios_inferidos_por_hora"
    },
    carpeta / "hgb_demanda_horaria.joblib"
)

print("Modelo guardado correctamente")

## 16. Alertas exploratorias

La regresión produce una cantidad estimada; aquí se transforma en una señal binaria para identificar horas con actividad real superior a 60 episodios. `precision` es la proporción de alertas predichas que fueron críticas, `recall` es la proporción de horas críticas detectadas y F1 es su media armónica. La matriz de confusión separa verdaderos negativos, falsos positivos, falsos negativos y verdaderos positivos.

Para el umbral 60, la salida observada reporta precision 0,517, recall 0,408 y F1 0,456, con matriz `[[7839, 254], [395, 272]]`. Son resultados de validación exploratoria, no una garantía de desempeño operacional ni una definición clínica de saturación.

La comparación posterior evalúa umbrales de predicción entre 40 y 60. El umbral 50 aparece como candidato exploratorio porque balancea las métricas en esta tabla, pero no es una regla operativa aprobada. La elección real requeriría costos de falsas alarmas, consecuencias de no detectar picos, revisión hospitalaria y validación prospectiva.


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

umbral = 60

real_critico = (val_ml["demanda"] > umbral).astype(int)
pred_critico = (pred_hgb > umbral).astype(int)

precision = precision_score(
    real_critico, pred_critico, zero_division=0
)

recall = recall_score(
    real_critico, pred_critico, zero_division=0
)

f1 = f1_score(
    real_critico, pred_critico, zero_division=0
)

print("Horas críticas reales:", real_critico.sum())
print("Horas críticas detectadas:", pred_critico.sum())
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))
print("\nMatriz de confusión:")
print(confusion_matrix(real_critico, pred_critico))

### Comparación exploratoria de umbrales

Se comparan umbrales 40, 45, 50, 55 y 60 manteniendo como evento real `demanda > 60`. En la salida observada, el umbral 40 alcanza recall 0,972 pero genera 1.573 falsas alarmas; el umbral 60 alcanza precision 0,517 pero deja 395 horas críticas no detectadas; el umbral 50 obtiene F1 0,540. Esta tensión ilustra que no existe un umbral óptimo sin definir previamente las prioridades operacionales.

En consecuencia, 50 debe describirse únicamente como candidato exploratorio del notebook, no como una alerta institucional aprobada. La integración debe separar claramente la predicción continua, el umbral configurable y la decisión asistencial.


In [ ]:
resultados_umbrales = []

for umbral_alerta in [40, 45, 50, 55, 60]:
    alerta = (pred_hgb > umbral_alerta).astype(int)

    resultados_umbrales.append({
        "umbral_alerta": umbral_alerta,
        "precision": precision_score(real_critico, alerta, zero_division=0),
        "recall": recall_score(real_critico, alerta, zero_division=0),
        "f1": f1_score(real_critico, alerta, zero_division=0),
        "falsas_alarmas": ((alerta == 1) & (real_critico == 0)).sum(),
        "horas_no_detectadas": ((alerta == 0) & (real_critico == 1)).sum()
    })

tabla_umbrales = pd.DataFrame(resultados_umbrales).round(3)

display(tabla_umbrales)

### Exportación de resultados para integración

La tabla exportada contiene fecha, demanda observada, predicción HGB, una alerta exploratoria con umbral 50, actividad alta real con umbral 60 y error absoluto. El CSV está orientado a análisis y visualización, por ejemplo en Power BI, y conserva tanto predicción como referencia retrospectiva.

Las predicciones históricas exportadas son resultados de validación retrospectiva de 2025; no son pronósticos futuros. Para operación real sería necesario ejecutar un flujo de inferencia con datos disponibles hasta el instante de predicción, monitorear calidad y versionar modelo, variables y umbral.


In [ ]:
from pathlib import Path

salida = val_ml[["fecha_hora", "demanda"]].copy()

salida["prediccion"] = pred_hgb
salida["alerta_actividad_alta"] = (
    salida["prediccion"] > 50
).astype(int)

salida["actividad_alta_real"] = (
    salida["demanda"] > 60
).astype(int)

salida["error_abs"] = abs(
    salida["demanda"] - salida["prediccion"]
)

carpeta = Path("resultados/predicciones")
carpeta.mkdir(parents=True, exist_ok=True)

archivo = carpeta / "validacion_demanda_2025.csv"

salida.to_csv(archivo, index=False)

print("Archivo guardado:", archivo)
print("Filas exportadas:", len(salida))

## 18. Limitaciones y trabajo futuro

Este notebook presenta una metodología exploratoria y sus resultados observados, no una validación de impacto hospitalario. La variable objetivo representa episodios inferidos a partir de OMOP, no ingresos confirmados; la reconstrucción de episodios requiere validación con el hospital. Además, los modelos presentan dificultades durante picos de actividad, especialmente por encima de 60 episodios, y las variables temporales solo son utilizables operacionalmente si existen registros recientes, completos y oportunos.

El conjunto de prueba de 2026 ya fue consultado durante el análisis, por lo que la comparación debe documentarse como retrospectiva y no como una evaluación futura completamente ciega. Aún no se ha demostrado impacto sobre tiempos de espera. Asimismo, cualquier simulación u optimización de recursos necesita incorporar restricciones, capacidades y costos hospitalarios verificados.

Como trabajo futuro, se requiere validar la definición de episodio, establecer un protocolo temporal de evaluación ciega, investigar métodos robustos para picos, monitorear deriva y disponibilidad de rezagos, y acordar con el hospital los costos de falsas alarmas y omisiones. Estas son líneas de investigación y no correcciones aplicadas en este notebook.

Con estas salvedades, la historia metodológica queda delimitada: problema y datos, análisis exploratorio, baseline, mejoras predictivas, interpretabilidad, evaluación segmentada, alertas exploratorias y persistencia para una posible integración analítica.
